# ALPD: Adaptive Latent Pruning Diffusion
**PeLAP-A — Preprint Prototype**

Pipeline:
1. Install dependencies
2. Copy source files from dataset → working dir
3. Sanity check
4. Train Baseline
5. Train ALPD
6. Ablation study
7. Evaluation
8. Visualization
9. Results summary

**Dataset layout expected:**
```
/kaggle/input/alpd-src/alpd/models.py
/kaggle/input/alpd-src/alpd/diffusion.py
/kaggle/input/alpd-src/alpd/train.py
/kaggle/input/alpd-src/alpd/evaluate.py
/kaggle/input/alpd-src/alpd/visualize.py
/kaggle/input/alpd-src/alpd/ablation.py
```

In [ ]:
# ── 1. Install dependencies ───────────────────────────────────
!pip install scipy --quiet
print('Done.')

In [ ]:
# ── 2. Copy source files from dataset → /kaggle/working ───────
import os, shutil

SRC = '/kaggle/input/alpd-src/alpd'   # correct path inside the dataset
DST = '/kaggle/working'

files = ['models.py', 'diffusion.py', 'train.py',
         'evaluate.py', 'visualize.py', 'ablation.py']

for fname in files:
    src = os.path.join(SRC, fname)
    dst = os.path.join(DST, fname)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Copied  {fname}')
    else:
        print(f'MISSING {src}')

In [ ]:
# ── 3. Sanity check ───────────────────────────────────────────
import sys
sys.path.insert(0, '/kaggle/working')

import torch
from models import ALPDModel
from diffusion import DDPMScheduler

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

model = ALPDModel(latent_ch=4, base_ch=64, use_pruning=True).to(device)
x_dummy = torch.randn(4, 3, 32, 32, device=device)
recon, z, mu, logvar = model.vae(x_dummy)
z_p, mask = model.get_mask(z)
print(f'z shape      : {z.shape}')
print(f'z_pruned shape: {z_p.shape}')
print(f'mask shape   : {mask.shape}')
print(f'mask values  : {mask[0].detach().cpu().numpy()}')

sched = DDPMScheduler(T=1000, device=device)
L_diff = sched.diffusion_loss(model.unet, z_p)
print(f'Diffusion loss (untrained): {L_diff.item():.4f}')
print('Sanity check PASSED ✓')

In [ ]:
# ── 4. Train BASELINE (~45-60 min on P100) ────────────────────
!python /kaggle/working/train.py \
    --mode baseline \
    --epochs 100 \
    --batch_size 128 \
    --lr 2e-4 \
    --latent_ch 4 \
    --base_ch 64 \
    --T 1000 \
    --data_dir /kaggle/working/data \
    --out_dir /kaggle/working/outputs \
    --save_every 10

In [ ]:
# ── 5. Train ALPD (~45-60 min on P100) ───────────────────────
!python /kaggle/working/train.py \
    --mode alpd \
    --epochs 100 \
    --batch_size 128 \
    --lr 2e-4 \
    --lambda_sparse 0.01 \
    --latent_ch 4 \
    --base_ch 64 \
    --T 1000 \
    --data_dir /kaggle/working/data \
    --out_dir /kaggle/working/outputs \
    --save_every 10

In [ ]:
# ── 6. Ablation study (5 λ × 30 epochs, ~60-90 min) ──────────
!python /kaggle/working/ablation.py \
    --lambdas 0.001 0.005 0.01 0.05 0.1 \
    --epochs 30 \
    --batch_size 128 \
    --latent_ch 4 \
    --base_ch 64 \
    --data_dir /kaggle/working/data \
    --out_dir /kaggle/working/outputs/eval

In [ ]:
# ── 7. Evaluation (FID, MSE, runtime, channel activity) ───────
!python /kaggle/working/evaluate.py \
    --baseline_ckpt /kaggle/working/outputs/baseline/checkpoints/best.pt \
    --alpd_ckpt     /kaggle/working/outputs/alpd/checkpoints/best.pt \
    --out_dir       /kaggle/working/outputs/eval \
    --data_dir      /kaggle/working/data \
    --n_samples     1000 \
    --diffusion_steps 200

In [ ]:
# ── 8. Generate publication figures ───────────────────────────
!python /kaggle/working/visualize.py \
    --alpd_ckpt       /kaggle/working/outputs/alpd/checkpoints/best.pt \
    --baseline_ckpt   /kaggle/working/outputs/baseline/checkpoints/best.pt \
    --eval_dir        /kaggle/working/outputs/eval \
    --history_alpd    /kaggle/working/outputs/alpd/history.json \
    --history_baseline /kaggle/working/outputs/baseline/history.json \
    --out_dir         /kaggle/working/outputs/figures \
    --data_dir        /kaggle/working/data

In [ ]:
# ── 9a. View figures inline ────────────────────────────────────
from IPython.display import Image, display
import glob, os

fig_dir = '/kaggle/working/outputs/figures'

# Convert PDFs to PNG for inline display (Kaggle doesn't render PDFs inline)
import subprocess
for pdf in sorted(glob.glob(f'{fig_dir}/*.pdf')):
    png = pdf.replace('.pdf', '.png')
    subprocess.run(['convert', '-density', '150', pdf, png],
                   capture_output=True)

for fig_path in sorted(glob.glob(f'{fig_dir}/*.png')):
    print(os.path.basename(fig_path))
    display(Image(fig_path))

pdfs = sorted(glob.glob(f'{fig_dir}/*.pdf'))
print(f'\nPDF figures for paper ({len(pdfs)} files):')
for p in pdfs:
    print(f'  {p}  ({os.path.getsize(p)//1024} KB)')

In [ ]:
# ── 9b. Final results summary ─────────────────────────────────
import json

with open('/kaggle/working/outputs/eval/eval_results.json') as f:
    r = json.load(f)

print('\n' + '='*55)
print(' ALPD vs BASELINE — FINAL RESULTS')
print('='*55)
print(f"{'Metric':<28} {'Baseline':>12} {'ALPD':>12}")
print('-'*55)
print(f"{'Parameters':<28} {r.get('baseline_params',0):>12,} {r.get('alpd_params',0):>12,}")
print(f"{'Recon MSE':<28} {r.get('baseline_recon_mse',0):>12.4f} {r.get('alpd_recon_mse',0):>12.4f}")
fid_b = r.get('baseline_fid', 'N/A')
fid_a = r.get('alpd_fid', 'N/A')
fid_b_str = f'{fid_b:.2f}' if isinstance(fid_b, float) else str(fid_b)
fid_a_str = f'{fid_a:.2f}' if isinstance(fid_a, float) else str(fid_a)
print(f"{'FID':<28} {fid_b_str:>12} {fid_a_str:>12}")
print(f"{'Runtime (s)':<28} {r.get('baseline_runtime_mean',0):>12.3f} {r.get('alpd_runtime_mean',0):>12.3f}")
print(f"{'Active Channels':<28} {'4 / 4':>12} {r.get('alpd_active_channels_mean',0):>10.1f} / 4")
print('='*55)